# ALICE Group Notebook: Hadron Species Ratios
## Measuring the QGP Through Its Particles

**Vanderbilt Programs for Talented Youth | Instructor: Jennifer James, Ph.D. Candidate**

---

### What does ALICE measure?

ALICE (A Large Ion Collider Experiment) at the LHC was specifically designed to study the QGP through the **identified particles** it produces. Not just "charged hadrons" like PHENIX measured, but individual species: pions (π), kaons (K), protons (p), and their antiparticles.

The **ratios** of these species tell us about the temperature and baryon density of the QGP fireball at the moment it "freezes out" back into ordinary hadrons — the **chemical freeze-out** temperature.

One of the most surprising discoveries at RHIC (and confirmed at LHC) was the **baryon anomaly**: at intermediate pT (2–6 GeV/c), the proton-to-pion ratio in central Au+Au is much larger than in p+p. This was totally unexpected and is still not fully understood.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(7)
print("Libraries loaded!")

---

## Part 1: Particle Production — Loading the Data

We use a toy model based on the **Boltzmann-Gibbs blast-wave** parametrization, which describes how particles are produced from a thermally equilibrated, radially expanding fireball. The key parameters are:

- **T_chem** — chemical freeze-out temperature (~156 MeV at LHC, close to the QCD crossover temperature)
- **β_T** — transverse flow velocity (the fireball expands and pushes particles outward)
- **Mass** — heavier particles (protons) get more of a kick from the flow

In [ ]:
# Particle masses (GeV/c2)
m_pi  = 0.1396   # pion (lightest hadron)
m_K   = 0.4937   # kaon
m_p   = 0.9383   # proton

pT = np.linspace(0.1, 6.0, 200)  # pT range in GeV/c

def blast_wave(pT, mass, T, beta, norm):
    """
    Simplified blast-wave spectrum.
    T     = temperature (GeV)
    beta  = mean transverse flow velocity (fraction of c)
    norm  = overall normalization
    """
    mT = np.sqrt(mass**2 + pT**2)   # transverse mass
    # Boltzmann-like with flow boost
    flow_boost = np.cosh(np.arctanh(beta))
    arg = mT * flow_boost / T
    # Bessel-function approximation: exp(-arg) * pT / mT
    spectrum = norm * pT / mT * np.exp(-arg)
    return spectrum

# ── p+p spectra: lower temperature, no collective flow ────────────────────────
T_pp, beta_pp = 0.095, 0.0   # no radial flow in p+p

spec_pi_pp = blast_wave(pT, m_pi, T_pp, beta_pp, norm=1200)
spec_K_pp  = blast_wave(pT, m_K,  T_pp, beta_pp, norm=180)
spec_p_pp  = blast_wave(pT, m_p,  T_pp, beta_pp, norm=95)

# ── Au+Au central: higher temperature, strong radial flow ─────────────────────
T_AuAu, beta_AuAu = 0.120, 0.62   # strong collective flow pushes heavy particles

spec_pi_AuAu = blast_wave(pT, m_pi, T_AuAu, beta_AuAu, norm=18000)
spec_K_AuAu  = blast_wave(pT, m_K,  T_AuAu, beta_AuAu, norm=3200)
spec_p_AuAu  = blast_wave(pT, m_p,  T_AuAu, beta_AuAu, norm=2800)

print("Blast-wave spectra computed.")
print(f"  p+p:    T = {T_pp*1000:.0f} MeV,  beta_T = {beta_pp:.2f}c")
print(f"  Au+Au:  T = {T_AuAu*1000:.0f} MeV,  beta_T = {beta_AuAu:.2f}c  (strong collective flow!)")

---

## Part 2: The pT Spectra — Seeing the Radial Flow Boost

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {'pi': 'steelblue', 'K': 'darkorange', 'p': 'firebrick'}

for ax, specs, title, T, beta in [
    (axes[0], (spec_pi_pp, spec_K_pp, spec_p_pp),       f'p+p  (T={T_pp*1000:.0f} MeV, no flow)',        T_pp,    beta_pp),
    (axes[1], (spec_pi_AuAu, spec_K_AuAu, spec_p_AuAu), f'Au+Au central  (T={T_AuAu*1000:.0f} MeV, β={beta_AuAu:.2f}c)', T_AuAu, beta_AuAu),
]:
    ax.semilogy(pT, specs[0], color=colors['pi'], linewidth=2.5, label=r'Pion ($\pi^+$,  m=140 MeV)')
    ax.semilogy(pT, specs[1], color=colors['K'],  linewidth=2.5, label=r'Kaon ($K^+$,   m=494 MeV)')
    ax.semilogy(pT, specs[2], color=colors['p'],  linewidth=2.5, label=r'Proton ($p$,    m=938 MeV)')
    ax.set_xlabel('Transverse momentum pT (GeV/c)', fontsize=12)
    ax.set_ylabel('Invariant yield (arb. units, log scale)', fontsize=12)
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3, which='both')
    ax.set_xlim(0, 6)

plt.suptitle('Identified hadron pT spectra: blast-wave model\n'
             'Notice how the proton spectrum changes shape in Au+Au vs p+p', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print("In p+p: pions always dominate at all pT (lightest = most produced).")
print("In Au+Au: at intermediate pT (2-4 GeV/c), the proton spectrum 'catches up'.")
print("This is the radial flow boost: heavier particles get pushed to higher pT.")

---

## Part 3: The Baryon Anomaly — Proton/Pion Ratio

In [ ]:
# Compute proton-to-pion ratios
ratio_pp   = spec_p_pp   / spec_pi_pp
ratio_AuAu = spec_p_AuAu / spec_pi_AuAu

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(pT, ratio_pp,   color='black',    linewidth=2.5, linestyle='--',
        label='p+p reference')
ax.plot(pT, ratio_AuAu, color='firebrick', linewidth=2.5,
        label='Au+Au central (0–10%)')

# Shade the anomaly region
anomaly_mask = (pT >= 1.5) & (pT <= 4.5)
ax.fill_between(pT[anomaly_mask], ratio_pp[anomaly_mask], ratio_AuAu[anomaly_mask],
                alpha=0.3, color='firebrick', label='Baryon anomaly region')

ax.axhspan(0, ratio_pp.max()*1.1, alpha=0.0)
ax.annotate('Baryon anomaly:\np/π ratio enhanced\nin Au+Au',
            xy=(2.8, ratio_AuAu[np.argmin(np.abs(pT-2.8))]),
            xytext=(3.8, 0.55),
            fontsize=11, color='firebrick',
            arrowprops=dict(arrowstyle='->', color='firebrick', lw=1.5))

ax.set_xlabel('Transverse momentum pT (GeV/c)', fontsize=13)
ax.set_ylabel('Proton / Pion yield ratio  p/π', fontsize=13)
ax.set_title('The Baryon Anomaly: proton/pion ratio in p+p vs Au+Au\n'
             'At intermediate pT, Au+Au produces relatively MORE protons than p+p', fontsize=11)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 6)
ax.set_ylim(0, 0.9)
plt.tight_layout()
plt.show()

# Find peak of anomaly
peak_idx = np.argmax(ratio_AuAu - ratio_pp)
print(f"Baryon anomaly peaks at pT ≈ {pT[peak_idx]:.1f} GeV/c")
print(f"  p/π in p+p:   {ratio_pp[peak_idx]:.3f}")
print(f"  p/π in Au+Au: {ratio_AuAu[peak_idx]:.3f}")
print(f"  Enhancement factor: {ratio_AuAu[peak_idx]/ratio_pp[peak_idx]:.1f}x")
print()
print("This ~3x enhancement was one of RHIC's most surprising discoveries.")
print("ALICE has confirmed and extended it at much higher LHC energies.")

---

## Part 4: Chemical Freeze-Out — Reading the QGP Temperature

The **ratios of particle species** (π, K, p, Λ, ...) at low pT are set by the **chemical freeze-out temperature** — the temperature at which the QGP stops exchanging particle identity and the species ratios lock in.

By fitting the measured ratios to a thermal model, physicists extract T_chem. At LHC energies, T_chem ≈ 156 MeV — remarkably close to the QCD crossover temperature predicted by lattice QCD calculations. This is one of the strongest pieces of evidence that the system reaches thermal equilibrium.

In [ ]:
# Thermal model: particle ratios as a function of temperature
T_range = np.linspace(0.100, 0.180, 100)  # GeV

# Simplified thermal ratios: K/pi and p/pi from grand canonical ensemble
# K/pi ~ exp(-(m_K - m_pi)/T) * (m_K/m_pi)^1.5
# p/pi ~ exp(-(m_p  - m_pi)/T) * (m_p /m_pi)^1.5
K_pi_thermal = np.exp(-(m_K - m_pi)/T_range) * (m_K/m_pi)**1.5 * 0.55
p_pi_thermal = np.exp(-(m_p - m_pi)/T_range) * (m_p/m_pi)**1.5 * 0.12

# ALICE measured values (approximate, from published data)
T_measured   = 0.1565   # GeV
K_pi_meas    = 0.155
p_pi_meas    = 0.047
K_pi_err     = 0.010
p_pi_err     = 0.005

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, thermal, meas, err, label, color in [
    (axes[0], K_pi_thermal, K_pi_meas, K_pi_err, 'K/π ratio', 'darkorange'),
    (axes[1], p_pi_thermal, p_pi_meas, p_pi_err, 'p/π ratio', 'firebrick'),
]:
    ax.plot(T_range*1000, thermal, color=color, linewidth=2.5, label='Thermal model')
    ax.axhline(y=meas, color='black', linestyle='--', linewidth=1.5,
               label=f'ALICE measured: {meas:.3f} ± {err:.3f}')
    ax.axhspan(meas-err, meas+err, alpha=0.2, color='black')
    # Find intersection
    idx = np.argmin(np.abs(thermal - meas))
    ax.axvline(x=T_range[idx]*1000, color=color, linestyle=':', linewidth=1.5,
               label=f'T_chem ≈ {T_range[idx]*1000:.0f} MeV')
    ax.set_xlabel('Temperature T (MeV)', fontsize=12)
    ax.set_ylabel(label, fontsize=12)
    ax.set_title(f'{label} vs freeze-out temperature', fontsize=11)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.suptitle('Chemical freeze-out temperature from particle ratios\n'
             'ALICE Pb+Pb at √s_NN = 2.76 TeV', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print(f"Chemical freeze-out temperature: T_chem ≈ {T_measured*1000:.0f} MeV")
print(f"QCD crossover temperature (lattice QCD): T_c ≈ 154-158 MeV")
print(f"These are remarkably close — the system freezes out right at the phase boundary!")

---

## Your Group's Checkpoint Questions

1. In the pT spectra plot, at what pT value do the proton and pion spectra roughly cross in Au+Au? Why doesn't this crossing happen in p+p?

2. The radial flow velocity β = 0.62c in central Au+Au. This means the fireball is expanding at 62% the speed of light when it freezes out. How does a heavier particle (like a proton) respond differently to this flow kick compared to a light particle (like a pion)?

3. The chemical freeze-out temperature T_chem ≈ 156 MeV is very close to the QCD phase transition temperature T_c ≈ 155 MeV. What does this tell you about *when* in the collision history the particle species ratios are set?

4. ALICE at the LHC collides lead nuclei (Pb+Pb) at √s_NN = 2.76 and 5.02 TeV — much higher than RHIC's 200 GeV. Would you expect the chemical freeze-out temperature to be higher or lower at LHC energies? *(Hint: it's surprisingly similar — this is an active area of research.)*